In [ ]:
!pip -q install roboflow imagehash

In [ ]:
import os, glob, shutil, yaml, random
import imagehash
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
from google.colab import userdata, drive
from roboflow import Roboflow

# ====== KONFIGURASI ======
SRC_WORKSPACE = "wotech-team"
SRC_PROJECT   = "deteksi-penyakit-kulit-2-kfwzc"
SRC_VERSION   = 4            # versi tanpa augmentasi ("7class Evaluasi")
DST_WORKSPACE = "wotech-team"
DST_PROJECT   = "deteksi-penyakit-kulit-jurnal-1-yolo11n-t1"
THRESH        = 5            # ambang jarak perceptual hash (0 = identik persis)
OUT_DIR       = "/content/drive/MyDrive/skin_runs"
# =========================

drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(SRC_WORKSPACE).project(SRC_PROJECT).version(SRC_VERSION).download("yolov11")
root = dataset.location
DATA = f"{root}/data.yaml"
print("Lokasi dataset:", root)

In [ ]:
def count_images():
    return {s: len(glob.glob(f"{root}/{s}/images/*")) for s in ["train", "valid", "test"]}

def class_table():
    names = yaml.safe_load(open(DATA))["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else names
    rows = {}
    for s in ["train", "valid", "test"]:
        cnt = Counter()
        for f in glob.glob(f"{root}/{s}/labels/*.txt"):
            for line in open(f):
                if line.strip():
                    cnt[int(line.split()[0])] += 1
        rows[s] = cnt
    t = pd.DataFrame({s: {names[c]: rows[s][c] for c in names} for s in rows})
    t["total"] = t.sum(axis=1)
    return t

print("Gambar per split (sebelum):", count_images())
print(class_table())

In [ ]:
def find_pairs(thresh=THRESH):
    items = []
    for split in ["train", "valid", "test"]:
        for p in glob.glob(f"{root}/{split}/images/*"):
            items.append((p, split, imagehash.phash(Image.open(p))))
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            d = items[i][2] - items[j][2]
            if items[i][1] != items[j][1] and d <= thresh:
                pairs.append((items[i], items[j], d))
    return pairs

pairs = find_pairs()
print("Total pasangan:", len(pairs))
print("Jenis pasangan:", Counter(tuple(sorted((a[1], b[1]))) for a, b, d in pairs))
print("Distribusi jarak:", sorted(Counter(d for _, _, d in pairs).items()))

In [ ]:
worst = sorted(pairs, key=lambda x: -x[2])[:8]
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax in axes.ravel():
    ax.axis("off")
for k, (a, b, d) in enumerate(worst):
    for n, it in enumerate((a, b)):
        ax = axes[(k * 2 + n) // 4][(k * 2 + n) % 4]
        ax.imshow(Image.open(it[0]))
        ax.set_title(f"{it[1]} (d={d})", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
BACKUP = "/content/dataset_backup"
if not os.path.exists(BACKUP):
    shutil.copytree(root, BACKUP)
    print("Backup dibuat di", BACKUP)

to_remove = {}
for a, b, d in pairs:
    if "train" in (a[1], b[1]):
        victim, keeper = (a, b) if a[1] == "train" else (b, a)
    else:
        victim, keeper = (a, b) if a[1] == "valid" else (b, a)
    to_remove[victim[0]] = (victim[1], keeper[0], keeper[1], d)

for p in to_remove:
    lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    for f in (p, lbl):
        if os.path.exists(f):
            os.remove(f)

for c in glob.glob(f"{root}/**/*.cache", recursive=True):
    os.remove(c)

log = pd.DataFrame(
    [(k, *v) for k, v in to_remove.items()],
    columns=["removed_file", "removed_split", "kept_file", "kept_split", "hash_dist"],
)
log.to_csv(f"{OUT_DIR}/duplicates_removed.csv", index=False)

print("Gambar dihapus:", len(to_remove))
print(log["removed_split"].value_counts())
print("Gambar per split (sesudah):", count_images())

In [ ]:
left = find_pairs()
print("Sisa pasangan duplikat:", len(left))
if left:
    print("Masih ada sisa. Jalankan ulang sel 4 dan sel 6 sampai hasilnya 0.")

t = class_table()
print(t)
t.to_csv(f"{OUT_DIR}/class_instances_clean.csv")

In [ ]:
def is_bg(p):
    lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    return (not os.path.exists(lbl)) or (not open(lbl).read().strip())

for s in ["train", "valid", "test"]:
    n_bg = sum(is_bg(p) for p in glob.glob(f"{root}/{s}/images/*"))
    print(f"{s}: {n_bg} gambar tanpa anotasi")

bg = [p for p in glob.glob(f"{root}/train/images/*") if is_bg(p)]
random.seed(0)
sample = random.sample(bg, min(12, len(bg)))
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax in axes.ravel():
    ax.axis("off")
for ax, p in zip(axes.ravel(), sample):
    ax.imshow(Image.open(p))
plt.tight_layout()
plt.show()

In [ ]:
zip_path = shutil.make_archive(f"{OUT_DIR}/dataset_bersih_yolov11", "zip", root)
print("Tersimpan:", zip_path)

In [ ]:
ws = rf.workspace(DST_WORKSPACE)
ws.upload_dataset(
    root,
    DST_PROJECT,
    num_workers=8,
    project_license="MIT",
    project_type="object-detection",
)
print("Upload selesai. Cek hasilnya di Roboflow.")